In [54]:
library(tidyverse)
library(lubridate)
library(readxl)


In [55]:
#loading all the dataset 
fertilizer    <- read_excel("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/CMO-Historical-Data-Monthly (1).xlsx", sheet = "Monthly Prices",skip = 4)
CPI           <- read.csv("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/CPI_DATA.csv")
crude_oil     <- read.csv("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/crude oil data.csv")
FX            <- read.csv("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/Exchange_Rate.csv")
fuel          <- read_csv("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/Fuel_Prices_combined.csv")


#standardising the date into a consistent format 
fertilizer <- mutate(fertilizer, clean_date = str_replace(as.character(Date), "M", "-") , Date = ym(clean_date))

CPI <- mutate(CPI, 
              start_month = str_extract(Western.Timeline, "(?<=Mid-)[A-Za-z]{3}"),
                date_str = paste("15",start_month,Year),
                 Date = dmy(date_str))
crude_oil <- mutate(crude_oil , Date = ymd(observation_date))
FX <- mutate(FX, Date= mdy(Date))
fuel <- mutate(fuel, Date = mdy(Date))


Rows: 170 Columns: 10-- Column specification --------------------------------------------------------
Delimiter: ","
chr (3): Date, Effective Date (BS), Effective Time
dbl (4): Petrol, Diesel, Kerosene, LPG
lgl (3): ATF (DP), ATF (DF), Remarks
i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.

In [56]:
head(crude_oil)

observation_date,DCOILBRENTEU,Date
2013-01-02,112.98,2013-01-02
2013-01-03,113.03,2013-01-03
2013-01-04,112.58,2013-01-04
2013-01-07,112.49,2013-01-07
2013-01-08,113.03,2013-01-08
2013-01-09,113.07,2013-01-09


In [57]:
fertilizer <- fertilizer %>% slice(-1)

brent_monthly <- crude_oil %>%
  mutate(Date = floor_date(Date, "month")) %>%
  group_by(Date) %>%
  summarise(brent = mean(DCOILBRENTEU, na.rm = TRUE))

In [58]:
#filtering the dataframe and choosing only required column
refined_fert <- select(fertilizer, Date, DAP, Urea)
refined_CPI <- select(CPI, Date,inflation_rate = `Y.o.Y.Inflation.Rate....`) %>%
    mutate(Date = floor_date(Date , "month"))

refined_crude_oil <- select( brent_monthly, Date, oil_price = brent)
refined_FX <- select(FX , Date, exchange_rate = Price)
refined_fuel<- select(fuel, Date, Diesel)



In [59]:
study_start <- as.Date("2013-07-01")
study_end   <- as.Date("2026-07-01")
df_list <- list(refined_fert,refined_CPI,refined_crude_oil,refined_FX)
macro_clean <- map(df_list, ~ filter(.x, Date >= study_start & Date <= study_end))
macro_csv <- reduce(macro_clean, full_join, by = "Date")
macro_csv <- arrange(macro_csv, Date)

In [60]:
write.csv(macro_csv,"E:/Users/BROTHER COMPUTER/Desktop/research/Cleaned_Data/macro_clean.csv")

In [67]:
kalimati <- read_csv("E:/Users/BROTHER COMPUTER/Desktop/research/Cleaned_Data/kalimati_monthly.csv") %>%
    mutate(Date = mdy(Date))
head(kalimati)
master_clean <- kalimati %>% left_join(macro_csv, by = "Date")
master_clean <- filter(master_clean ,  Date >= study_start & Date <= study_end)
master_clean <- master_clean %>%
  mutate(
    # war dummies
    war_ru = if_else(Date >= as.Date("2022-02-01"), 1, 0),
    war_ii = if_else(Date >= as.Date("2024-04-01"), 1, 0),
    covid  = if_else(Date >= as.Date("2020-03-01") &
                     Date <= as.Date("2021-12-31"), 1, 0),

    # ITS time variables
    time = as.numeric(Date - min(Date)) / 30,  # months from start

    time_after_ru = if_else(Date >= as.Date("2022-02-01"),
                    as.numeric(Date - as.Date("2022-02-01")) / 30, 0),

    time_after_ii = if_else(Date >= as.Date("2024-04-01"),
                    as.numeric(Date - as.Date("2024-04-01")) / 30, 0)
  )
master_clean <- arrange(master_clean, Date)
write.csv(master_clean,"E:/Users/BROTHER COMPUTER/Desktop/research/Cleaned_Data/master_clean.csv", row.names = FALSE )

Rows: 813 Columns: 3-- Column specification --------------------------------------------------------
Delimiter: ","
chr (2): commodity, Date
dbl (1): monthly_average
i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.

commodity,Date,monthly_average
Cabbage(Local),2013-06-01,11.13636
Cabbage(Local),2013-07-01,16.20000
Cabbage(Local),2013-08-01,35.71429
Cabbage(Local),2013-09-01,42.36000
Cabbage(Local),2013-10-01,32.15385
Cabbage(Local),2013-11-01,42.30435
